# 04 · 효과·불확실성·배포 판단

**핵심 실행 12분** · 해결 확인율 상승과 지연 악화가 공존할 때 근거를 설명합니다. 사용자 효과 해석을 연습하는 수업 파일입니다.

준비: 저장소 터미널에서 `python -m pip install -r requirements/week12-online.txt` 실행.
VS Code 오른쪽 위 **Select Kernel → 저장소 .venv의 Python 3.11**을 선택합니다.
다른 Notebook의 변수나 실행 순서를 요구하지 않습니다. 이 파일의 셀은 위에서 아래로 실행하세요.

**출처 구분:** CSV는 교수 작성 합성 사용자 데이터입니다. FastAPI 요청은 실제 실습 서비스 실행이며 실제 고객 실험이 아닙니다.
05번의 Ollama만 실제 로컬 모델을 호출합니다. Docker·클라우드 계정·유료 API는 필수가 아닙니다.

## 코드 셀 실행 지도

| 코드 셀 | 실행 | 확인할 결과 |
|---:|---|---|
| 1 | 환경 준비 | Python 3.11과 설정 버전 |
| 2 | 수업 사례 불러오기 | A 55.02%, B 64.04%, B p95 2867.73ms, HOLD. |
| 3 | 절대 차이와 상대 개선 | 절대 차이 +9.02%p / 상대 개선 +16.39%. |
| 4 | 95% 구간 직접 확인 | 95% Newcombe–Wilson 구간 [5.10, 12.91]%p, 하한>0 assert 통과. |
| 5 | 판단 조건의 순서 | healthy REVIEW / slow HOLD / srm INVALID. |
| 6 | 표본·기간과 중간 결과 | 2일 결과 WAIT / 관찰 창 미완료 WAIT / 지연 위험 HOLD. |
| 7 | 작은 효과와 필요한 표본 · 심화 | 목표 차이 10%p→376명, 5%p→1534명, 2.5%p→6180명. 각각 버전당 인원. |
| 8 | 매일 유의성 확인 후 종료 · 심화 | seed=12, 500회: 종료 시점 한 번 3.6%, 14번 중 한 번이라도 17.6% 거짓 양성. |
| 9 | 내 판단 근거 저장 | reports/my-slow-result.json 생성, HOLD 판단 예시 출력. |

**이번 파일의 역할:** 유효한 사용자 비교 결과를 효과·불확실성·운영 보호 기준으로 해석합니다. 수업의 핵심 사례이며, 11주차 문항별 답변 점수를 사용자 성과로 바꾸어 부르지 않습니다.

**재실행 방법:** 셀 1–6·9가 핵심, 7–8은 선택 심화입니다. 셀 9는 my-slow-result.json을 갱신하므로 자신의 수정 파일은 다른 이름으로 보관하세요.

**코드 셀 1 설명:** 실험을 비교하기 전에 실행 환경과 조건부터 맞춥니다. 파일별 커널은 독립적이므로 각 Notebook의 첫 코드 셀부터 실행합니다.

번호는 코드 셀 순서이며 실행 횟수 `[1]`, `[2]`와 다릅니다. [셀별 교수 설명](https://smilesjcha.github.io/llmops-16week/week12/lecture/instructor/#book-04)

In [ ]:
from pathlib import Path
import sys, json, os
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "week12/lab/analysis.py").exists()), None)
if ROOT is None:
    raise RuntimeError("저장소 폴더를 VS Code로 열고 실행하세요.")
if sys.version_info[:2] != (3, 11):
    raise RuntimeError("저장소 .venv의 Python 3.11 커널을 선택하세요.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import display, HTML
from week12.lab.experiment import DEFAULT, assign
from week12.lab.analysis import analyze
from week12.lab.simulate import load_units
print("준비 완료:", sys.version.split()[0], "실험 설정", DEFAULT.version)


## 코드 셀 2 · 수업 사례 불러오기

slow는 정상 A/B 배정이지만 B의 지연이 큰 사례입니다. 여기서 구한 수치로 판단 원리를 연습합니다.

In [ ]:
data = load_units("slow")
report = analyze(data)
display(pd.DataFrame(report["arms"]).T)
print(report["decision"], report["reasons"])

**확인과 해석:** A 55.02%, B 64.04%, B p95 2867.73ms, HOLD.

**화면에서 볼 곳:** rate·p95_ms·reasons

**설명 핵심:** 이익과 운영 위험은 별개입니다. p95는 응답 시간의 95번째 백분위수이며, 이번 사례에서 B가 사전에 정한 1800ms 상한을 넘었기 때문에 보류합니다.

**직접 설명:** 해결 확인율이 약 9%p 높아져도 배포를 보류하는 이유는 무엇인가요?

## 코드 셀 3 · 절대 차이와 상대 개선

퍼센트포인트(%p)와 상대 비율(%)을 구분합니다. 같은 결과를 두 방식으로 표현할 수 있습니다.

In [ ]:
a,b = report["arms"]["A"],report["arms"]["B"]
absolute = b["rate"]-a["rate"]
relative = absolute/a["rate"]
print(f"절대 차이 {100*absolute:.2f}%p / 상대 개선 {100*relative:.2f}%")

**확인과 해석:** 절대 차이 +9.02%p / 상대 개선 +16.39%.

**화면에서 볼 곳:** (B−A)와 (B−A)/A

**설명 핵심:** 같은 결과를 다른 기준으로 표현한 값입니다. 퍼센트포인트는 비율의 차이, 상대 개선율은 A를 기준으로 얼마나 늘었는지입니다. 단위를 빼고 더 큰 숫자만 강조하지 않습니다.

**직접 설명:** 9%와 9%p를 구분하지 않으면 어떤 오해가 생기나요?

## 코드 셀 4 · 95% 구간 직접 확인

독립 사용자의 이진 지표에 Newcombe–Wilson 방법을 사용합니다. 확률 95%로 특정 참값이 있다는 뜻으로 해석하지 않습니다.

In [ ]:
from week12.lab.analysis import difference_interval
interval = difference_interval(a["resolved"],a["n"],b["resolved"],b["n"])
print({k:round(100*interval[k],2) for k in ["difference","low","high"]}, "%p")
assert interval["low"] > 0

**확인과 해석:** 95% Newcombe–Wilson 구간 [5.10, 12.91]%p, 하한>0 assert 통과.

**화면에서 볼 곳:** difference·low·high

**설명 핵심:** 독립 사용자와 고정 시점 가정에서 차이의 불확실성을 봅니다. 같은 절차를 반복했을 때 구간의 약 95%가 참값을 포함하도록 만든 방법이며, 양의 구간도 지연 문제를 해결해주지는 않습니다.

**직접 설명:** 이 구간이 양수라는 사실만으로 지연 문제를 무시해도 될까요?

## 코드 셀 5 · 판단 조건의 순서

데이터 불량 → 해석 중단, 위험 → 보류, 계획 미달 → 대기, 효과 불확실 → 결론 유보, 통과 → 사람 검토입니다.

In [ ]:
results = {name: analyze(load_units(name)) for name in ["healthy","slow","srm"]}
display(pd.DataFrame([{"사례":name,"판단":r["decision"],"이유":" / ".join(r["reasons"])} for name,r in results.items()]))
print("healthy라도 REVIEW이지 자동 배포가 아닙니다.")

**확인과 해석:** healthy REVIEW / slow HOLD / srm INVALID.

**화면에서 볼 곳:** 사례별 판단 이유

**설명 핵심:** REVIEW는 자동 승인, HOLD는 실행 오류라는 뜻이 아닙니다. REVIEW는 사람 검토 대기, HOLD는 운영 위험 때문에 보류, INVALID는 비교 근거부터 복구해야 한다는 뜻입니다.

**직접 설명:** 11주차 HOLD·REVIEW에 12주차 WAIT·INVALID가 추가된 이유는 무엇인가요?

## 코드 셀 6 · 표본·기간과 중간 결과

효과가 보이기 시작했다는 이유로 종료일을 당기지 않습니다. 운영 장애로 중단하는 것과 성과 승리 선언을 구분합니다.

In [ ]:
valid = load_units("healthy")
print("2일 중간 결과:",analyze(valid,elapsed_days=2)["decision"])
print("관찰 창 미완료:",analyze(valid,outcomes_mature=False)["decision"])
print("지연 위험 조기 중단:",analyze(data,elapsed_days=2)["decision"])

**확인과 해석:** 2일 결과 WAIT / 관찰 창 미완료 WAIT / 지연 위험 HOLD.

**화면에서 볼 곳:** elapsed_days·outcomes_mature

**설명 핵심:** 효과가 좋아 보인다는 이유로 계획을 일찍 끝내지 않습니다. 반면 위험을 줄이기 위한 운영 중단은 최종 효과 판단을 기다릴 필요가 없습니다.

**직접 설명:** 왜 성과 판단은 기다리지만 심한 오류나 지연은 바로 중단할 수 있나요?

## 코드 셀 7 · 작은 효과와 필요한 표본 · 심화

이진 지표·50:50 배정·양측 5%·검정력 80%의 근사 계획입니다. 요청 수가 아니라 독립 사용자 수입니다.

In [ ]:
from week12.lab.analysis import sample_size
display(pd.DataFrame([{"목표 절대 차이":f"{100*d:.1f}%p","필요 사용자/버전":sample_size(.55,d)} for d in [.10,.05,.025]]))

**확인과 해석:** 목표 차이 10%p→376명, 5%p→1534명, 2.5%p→6180명. 각각 버전당 인원.

**화면에서 볼 곳:** 필요 사용자/버전, 검정력 80% 가정

**설명 핵심:** 작은 개선일수록 더 많은 독립 사용자가 필요합니다. 학생 프로젝트에 사용자가 부족하다면 사용성 관찰이나 실패 사례 검증으로 범위를 정직하게 줄입니다.

**직접 설명:** 사용자 20명으로 작지만 유용한 효과를 확정하기 어려운 이유는 무엇인가요?

## 코드 셀 8 · 매일 유의성 확인 후 종료 · 심화

실제 효과가 없는 가상 세계를 반복합니다. 매일의 일반 p-value를 보고 유의하면 멈추는 규칙은 고정 시점 보장을 유지하지 않습니다.

In [ ]:
from week12.lab.analysis import peeking_demo
peek = peeking_demo()
display(pd.Series(peek))

**확인과 해석:** seed=12, 500회: 종료 시점 한 번 3.6%, 14번 중 한 번이라도 17.6% 거짓 양성.

**화면에서 볼 곳:** fixed_end와 any_day의 차이

**설명 핵심:** 효과가 없는 세계에서도 매일 유의하면 종료하는 규칙은 우연한 승리를 더 자주 만듭니다. 수업 코드의 한 시뮬레이션 결과이며 이 수치가 항상 같은 법칙은 아닙니다. 순차 추론은 별도 설계가 필요합니다.

**직접 설명:** 순차 검정은 매일 일반 검정을 반복하는 것과 무엇이 달라야 할까요?

## 코드 셀 9 · 내 판단 근거 저장

숫자 하나보다 결론·근거·다음 행동을 남깁니다. 복습 메모는 한 페이지로 정리합니다.

In [ ]:
out = ROOT/"week12/lab/reports"
out.mkdir(exist_ok=True,parents=True)
(out/"my-slow-result.json").write_text(json.dumps(report,ensure_ascii=False,indent=2))
decision_note = {"결론":"HOLD", "주요 근거":"B 해결 확인율 상승, p95 1800ms 기준 위반", "다음 변경":"출력 길이를 줄이고 오프라인 회귀 평가 후 새 실험", "한계":"합성 데이터이며 실제 제품 성과가 아님"}
display(pd.Series(decision_note))
print(out/"my-slow-result.json")

**확인과 해석:** reports/my-slow-result.json 생성, HOLD 판단 예시 출력.

**화면에서 볼 곳:** 결론·근거·다음 변경·한계

**설명 핵심:** 복습에서는 이 숫자로 자신의 판단을 적습니다. 출력 길이를 줄인다면 기간·배송비·재고 조건을 놓치지 않는지 오프라인 검증 후 새 사용자 실험으로 연결합니다.

**직접 설명:** 답변 길이를 줄인 다음 11주차의 어떤 검사를 다시 실행해야 할까요?

## 복습과 다음 단계

결과를 맞추기 위해 p95 기준을 3000ms로 바꾸지 마세요. 변경안의 품질·속도를 다시 확인한 뒤 새 실험으로 검증합니다. 11월 20일 개정 과제는 별도 Langfuse 미션 Notebook에서 진행합니다. 이 파일의 slow 분석은 사용자 효과 판단의 수업 예시입니다.

실행 오류는 Kernel을 다시 시작한 뒤 위에서부터 실행해 확인합니다. `ModuleNotFoundError`는 설치 터미널과 Notebook 커널이 같은 `.venv`인지 확인하세요.
실험 결과는 `week12/lab/reports/`에 저장하고 공개 GitHub에 학생 답안·개인정보를 올리지 않습니다.
[실행 안내](../WALKTHROUGH.md) · [100분 강의안](https://smilesjcha.github.io/llmops-16week/week12/lecture/) · [9–12주차 과제](https://smilesjcha.github.io/llmops-16week/week12/assignment/)
